# Segmentation workflow for multichannel images

This notebook demonstrates a complete workflow for **segmenting** nuclei, cytoplasm, and intracellular structures from 3D microscopy data, and for **quantifying** structures at both the object and cell level. The pipeline is modular and extensible, allowing different segmentation strategies depending on the channel and biological question.

The general pipeline includes:

1. Loading and preprocessing volumetric data
    - Intensity normalization
    - Optional downsampling and smoothing
    

2. Segmentation of cellular compartments
    - Nuclei (using deep learning models: StarDist or Cellpose)

    - Cytoplasm (via intensity- or membrane-based watershed)

    - Detection of intracellular structures

        - AICS segmentation workflows (e.g., dot/filament filters)
        

3. (optional) Post-processing (object removal, smoothing)

4. Quantification of structures per cell

    - Mapping detected objects to their parent cell

    - Extracting per-object and per-cell features (count, volume, intensity)

    - Exporting results to CSV for downstream analysis
    

5. Quality control

    - Visual overlays (e.g., maximum intensity projections, per-cell structure maps)

The goal is to link each detected intracellular structure to its cell of origin, enabling biologically meaningful, cell-level measurements.


---

### Set-up paths and loading scripts

In this section we define the **working environment** for the notebook. The main goals are:
    
1. Ensure reproducible path handling across different systems (local machine vs HPC).

    `cwd` is set to the current working directory, and the project root is assumed to be the parent folder
    
    
2. Add project root to sys.path so that custom scripts can be imported easily

     This makes sure Python knows where to look for the helper modules (`pre_processing`, `segment_3d`, `io_utils`, `quantification`)
     
     
3. Import helper functions form the scripts folder

#### Config File
Key parameters are controlled by a config file (`configs/segmentation_config.yml`). This means we can control key options without finding relevant parts of the code, which can be neater and more accessible for people limited coding experience. Familiarise yourself with the config file and how it is used if you want to understand the structure of the pipeline.

In [ ]:
## setting paths 
from pathlib import Path  ### switching to pathlib path-handling instead of the os - should work consistently between HPC and local machine
import sys

# current working directory of the notebook / script
cwd = Path.cwd()

# assume "scripts" folder is two levels up from cwd
project_root = cwd.parent.resolve().parent.resolve()

if str(project_root) not in sys.path:
    print("Adding to sys.path:", project_root)
    sys.path.append(str(project_root)) # add root to Python path (as a string) for finding scripts modules further


## Read in config
import os, yaml

config_file = os.path.join(project_root, "analysis_3d/configs/segmentation_config.yml")

with open(config_file, "r") as f:
    config = yaml.safe_load(f) 

## Set key variables
input_folder = os.path.join(project_root, config["input"]["input_folder"])
output_folder = os.path.join(project_root, config["output"]["results_dir"])
Path(output_folder).mkdir(parents=True, exist_ok=True)

## load libraries
import matplotlib.pyplot as plt
import numpy as np



In [ ]:
## loading scripts and helper functions
from analysis_3d.scripts.pre_processing import preprocess_3d_image ### pre-processing function (normalisation + optional downsampling)
from analysis_3d.scripts.segment_3d import segment_with_stardist, segment_with_cellpose, segment_cytoplasm ### StarDist3D (3d_demo) segmentation model - light and relatively quick to run
from analysis_3d.scripts.io_utils import load_multichannel_images, save_segmentation_results
from analysis_3d.scripts.quantification import quantify_objects, sphere_volume_um3, tidy_mask, filter_mask_by_size, quantify_structures_per_cell


### Input data setup
We specify the **input folder** containing raw images in the config file (`input_folder`).

1. Define a dictionary (*channel_map*) to tell the pipeline which channel corresponds to which biological compartment:

    - `"nucleus": 0` → channel index 0 contains the nuclear stain.

    - `"cytoplasm": 2` → channel index 2 contains cytoplasmic signal.

    - `"intracellular": 1` → channel index 1 contains the organelle/structure of interest.

    This mapping depends on the acquisition setup and might change between datasets — check the raw image metadata if in doubt (e.g. via opening a representative example image in ImageJ)

2. Make sure voxel size (`voxel_size_um`) is set correctly for the dataset, since this is required for volume and size calculations
3. Finally, we use `load_multichannel_images()` to read the dataset into memory

    - Returns a list of volumes, each represented as a dictionary with:

        - "filename" → original file name.

        - "channels" → dictionary of channels as (Z, Y, X) arrays.

    - The preview (print) shows how to access data for a single image (e.g. `all_volumes[0]["channels"]["nucleus"]`).

In [ ]:
# 1. load images as multichannel dictionary

# add a dictionary for your images, stating which channel corresponds to which of the structures
channel_map = {
    "nucleus": 0,
    "cytoplasm": 2,
    "intracellular": 1
}

# enter voxel size of your images (obtained from the metadata)
voxel_size_um = [float(config["input"]["voxel_size"]["z"]),
                 float(config["input"]["voxel_size"]["y"]),
                 float(config["input"]["voxel_size"]["x"])] # MUST BE IN Z, Y, X

# loading images from the input folder directory
all_volumes = load_multichannel_images(input_folder, channel_map)

# preview loaded images: first image [0] as example
print(all_volumes[0]["filename"])       # e.g., "sample01.tif"
print(all_volumes[0]["channels"].keys()) # what channels are included in the dictionary? should be: dict_keys(['nucleus', 'cytoplasm', 'intracellular'])
print(all_volumes[0]["channels"]["nucleus"].shape)  # (Z, Y, X)

#### Quick check: middle slice for each channel  

Here we display the **middle Z-slice** of each channel from the same (first) image of the folder. You can view different images by editing `Volume_number`.

This allows you to quickly confirm:  
- Channel order (e.g., nucleus, cytoplasm, intracellular)
- The objects in the different channels should overlap, with nucleus and intracellular objects within the cytoplasm area.
- That the data is loaded correctly and matches expectations  

If the signal looks wrong (e.g., nucleus is empty but cytoplasm is bright), check the `channel_map` definition above.

In [ ]:
### Which volume to view?
Volume_number = 0

# take the first loaded volume
vol = all_volumes[Volume_number]["channels"]

# find middle slice
z_mid = next(iter(vol.values())).shape[0] // 2  

# plot all channels in a grid
n_channels = len(vol)
fig, axes = plt.subplots(1, n_channels, figsize=(5 * n_channels, 5))

for ax, (name, data) in zip(axes, vol.items()):
    ax.imshow(data[z_mid], cmap="gray")
    ax.set_title(f"{name} (z={z_mid})")
    ax.axis("off")

plt.tight_layout()
plt.show()


### MAIN: per-channel preprocessing, segmentation, and quantification

This is the main section of the pipeline, which processes 3D microscopy images by segmenting different cellular compartments (**nucleus**, **cytoplasm**, and **intracellular structures** of interest) and quantifying structural features per cell. Each channel is preprocessed and segmented in a separate sub-section, allowing to save and visualise intermediate steps for quality checks.

---

**Global Parameters**

- **Output directory:** Defines where segmentation masks and overlays are saved.  
- **Number of images to test (`num_test_volumes`):** Defines how many test images from the input_folder will be used for the trial analysis throughout
- **Downsampling (`downsize_factor`):** Reduces memory usage and speeds up computation.  
- **Preprocessing:** Gaussian filtering and normalization (either per-slice or across the full 3D stack).  


In [ ]:
### params ###
num_test_volumes = config["input"]["num_test_images"]  # number of volumes from `frames` to segment

### pre-processing params ###
downsize_factor = config["preprocessing"]["downsize_factor"]   # scaling factor or 1 to keep original                            
per_slice_norm = config["preprocessing"]["per_slice_norm"]        # True = normalize per-slice, False = normalize whole stack

#### a) nucleus

This section describes the preprocessing, segmentation, and quantification of nuclei in 3D microscopy images. The workflow uses optionally either **StarDist** or  **Cellpose** models for segmentation, with post-processing and quality control steps available.

 1. **Preprocessing**: normalize nucleus channel, apply Gaussian filter to reduce noise, optionally downsample.  

 2. **Segmentation**
  - Segmentation model for nuclei:
    - `StarDist` (probability-based segmentation, very light, works best with more round-shape nuclear objects): controlled by probability (`prob_thresh`) and overlap (`nms_thresh`) thresholds.  
    - `Cellpose` (SAM-based segmentation, preferrably should be run on GPU)    
  - Output: **labeled 3D mask** where each nucleus is assigned a unique label.  

 3. **Quality Control (QC)**
- Save `.tif` mask stack and `.png` overlays (maximum-intensity projections with labels).  
- View some masks in the cell output to quickly check performance.


In [ ]:
# 2. nucleus segmentation

##### params #####
gaussian_nucleus = config["preprocessing"]["gaussian_nucleus"]  # apply gaussian filter to nuclei channel
sigma_um_nucleus = [float(v) for v in config["preprocessing"]["gaussian_sigma_nucleus"] ] # value of sigma for gaussian - could be adjusted depending on the data
use_model = config["segmentation"]["use_model"]  # "cellpose" or "stardist"

prob_thresh = config["segmentation"]["prob_thresh"]  # stardist param: sets the minimum confidence for accepting a predicted nucleus
nms_thresh = config["segmentation"]["nms_thresh"]    # stardist param: remove detections overlapping by more than this threshold

gpu= config["segmentation"]["gpu"]           # cellpose param: running without GPU is very slow

##################

results_dict = {}

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name
    
    # --- 2.1 nuclei preprocessing ---
    nuclei_volume = v["channels"]["nucleus"]
    nuclei_norm = preprocess_3d_image(nuclei_volume, 
                                      downsize_factor=downsize_factor,
                                      apply_gaussian_filter=gaussian_nucleus,           
                                      voxel_size_um=voxel_size_um,
                                      sigma_um=sigma_um_nucleus)             # could be adjusted depending on the data
    
    # --- 2.2 nuclei segmentation ---
    if use_model == "stardist":
        nuclei_mask = segment_with_stardist(nuclei_norm, prob_thresh=prob_thresh, nms_thresh=nms_thresh)
    elif use_model == "cellpose":
        nuclei_mask = segment_with_cellpose(nuclei_norm, gpu=gpu)
    
    # saving masks in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist

    results_dict[file_name]["nuclei_mask"] = nuclei_mask 
    
    # 2.3 save nuclei mask (.tiff) as well as PNG overlay image for quality check
    save_segmentation_results(
        nuclei_norm, 
        nuclei_mask, 
        output_root=output_folder, 
        experiment_label=f"{file_name}_nuclei",
        save_overlay=True
    )


In [ ]:
### View some masks - are they sensible?
# up to 5 pairs of (index, fname)
pairs = list(enumerate(results_dict.keys()))[:5]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 5))
if len(pairs) == 1:
    axes = [axes]

for ax, (idx, fname) in zip(axes, pairs):
    vol = all_volumes[idx]["channels"]
    mask = results_dict[fname]["nuclei_mask"]

    if len(vol.keys()) == 3:
        mid = len(vol["nucleus"]) // 2
        img = vol["nucleus"][mid]
        mask_slice = mask[mid]
    else:
        img = vol
        mask_slice = mask

    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(mask_slice == 0, mask_slice),
              cmap="autumn", alpha=0.5)
    ax.set_title(fname)
    ax.axis("off")

plt.tight_layout()
plt.show()

 4. **Quantification** (optional)
- Extract per-object features such as `count` and `volume`.  
- Compare measured nucleus volumes against expected biological range (e.g. 10 µm diameter sphere).  

In [ ]:
# 2.4 (optional) quantify nuclei

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name
    
    # --- retrieve nuclei masks ---
    nuclei_mask = results_dict[file_name]["nuclei_mask"]
    nuclei_mask = tidy_mask(nuclei_mask)
    
    # --- quantify nuclei ---
    df, summary = quantify_objects(nuclei_mask, voxel_size=voxel_size_um, features=("count","volume"))
    print(summary)


In [ ]:
# 2.4 example thresholds:
expected_d_um_nuclei = config["expected_d_um_nuclei"]     # e.g., 10 µm nuclei
tolerance_size_nuclei = config["tolerance_size_nuclei"]                     # ±25% tolerance

min_um3 = sphere_volume_um3(expected_d_um_nuclei * (1 - tolerance_size_nuclei))
max_um3 = sphere_volume_um3(expected_d_um_nuclei * (1 + tolerance_size_nuclei))
print("Expected nuclei volume:", min_um3, "to", max_um3, "µm3")

 5. **Filtering** (optional)
- Remove objects outside expected size tolerance.  
- Update results dictionary with filtered masks.

In [ ]:
# 2.5 (optional) filtering out nuclei with unwanted size
filter_nuclei = config["filter_nuclei"]     # True of False

if filter_nuclei:
    for v in all_volumes[:num_test_volumes]:
        file_name = v['filename']  # preserve original file name

        # step 1: get mask
        nuclei_mask = results_dict[file_name]["nuclei_mask"]
        nuclei_mask = tidy_mask(nuclei_mask)

        # step 2: filter mask
        nuclei_mask_filtered, df = filter_mask_by_size(nuclei_mask, expected_d_um_nuclei, tolerance_size_nuclei, voxel_size=voxel_size_um, return_df=True)
        
        n_before = len(df)
        n_after = df["keep"].sum() if len(df) > 0 else 0
        
        # step 3: replace mask in results dict for downstream use
        results_dict[file_name]["nuclei_mask"] = nuclei_mask_filtered

        print(f'{file_name}:',"Before filtering:", n_before)
        
        print("After filtering:", n_after)

---
#### b) cytoplasm

1. **Preprocessing**: normalise cytoplasm channel, Gaussian filtering typically disabled in `"intensity"` mode  

2. **Segmentation**
- Cytoplasm segmented via **watershed**, seeded by nucleus masks.  
- Two strategies available:  
  - `"intensity"` (default, based on cytoplasmic intensity)  
  - `"membrane"` (alternative, based on membrane-labeled channels)  

3. **Saving Results**: store cytoplasm mask in results dictionary, save `.tif` mask and `.png` overlay for QC

In [ ]:
# 3. cytoplasm segmentation
gaussian_cytoplasm = config["preprocessing"]["gaussian_cytoplasm"]  # apply gaussian filter to nuclei channel
gaussian_sigma_cytoplasm = config["preprocessing"]["gaussian_sigma_cytoplasm"]  # value of sigma for gaussian - cou

cytoplasm_mode = config["segmentation"]["cytoplasm_mode"]  # # Threshold for membrane signal to act as stopping boundary.
membrane_threshold = config["segmentation"]["membrane_threshold"]  #  "membrane" for boundary-based segmentation (membrane marker), "intensity" for intensity-based segmentation (e.g. cytoplasmic marker).

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name

    # get the corresponding nuclear mask
    if file_name not in results_dict:
        raise KeyError(f"Nuclear mask for {file_name} not found in previous results.")
    nuclei_mask = results_dict[file_name]["nuclei_mask"]

    # --- 3.1 cytoplasm channel preprocessing ---
    cytoplasm_volume = v["channels"]['cytoplasm']
    cytoplasm_norm = preprocess_3d_image(
            cytoplasm_volume, 
            downsize_factor=downsize_factor,
            apply_gaussian_filter=gaussian_cytoplasm,         # works better for 'intensity' mode of cytoplasm segmentation
            voxel_size_um=voxel_size_um,
            sigma_um=gaussian_sigma_cytoplasm      
            )
    
    # --- 3.2 cytoplasm segmentation (watershed) ---
    cytoplasm_mask = segment_cytoplasm(
        nuclei_mask,
        cytoplasm_norm, 
        mode=cytoplasm_mode,  # could also be "membrane" for alternative approach
        membrane_threshold=membrane_threshold
    )
    
    # 3.3 saving masks in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist
    
    results_dict[file_name]["cytoplasm_mask"] = cytoplasm_mask


    # saving results into a mask (.tiff) as well as PNG overlay image for quality check
    save_segmentation_results(
        cytoplasm_norm, 
        cytoplasm_mask, 
        output_root=output_folder, 
        experiment_label=f"{file_name}_cytoplasm",
        save_overlay=True
    )

In [ ]:
### View some masks - are they sensible?
# up to 5 pairs of (index, fname)
pairs = list(enumerate(results_dict.keys()))[:5]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 5))
if len(pairs) == 1:
    axes = [axes]

for ax, (idx, fname) in zip(axes, pairs):
    vol = all_volumes[idx]["channels"]
    mask = results_dict[fname]["cytoplasm_mask"]

    if len(vol.keys()) == 3:
        mid = len(vol["cytoplasm"]) // 2
        img = vol["cytoplasm"][mid]
        mask_slice = mask[mid]
    else:
        img = vol
        mask_slice = mask

    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(mask_slice == 0, mask_slice),
              cmap="autumn", alpha=0.5)
    ax.set_title(fname)
    ax.axis("off")

plt.tight_layout()
plt.show()

---
#### c) organelle / intracellular structure

This sub-section deals with segmenting/quantifying intracellular structures per cell mask, mapping the mask from intracellular channel to the cell_ID label from cytoplasmic mask. The general overview of the process is as follows: 

4.1 **Preprocessing**
- 4.0: obtain the suggested parameters for the preprocessing (from `suggest_normalization_param` helper function)
- Apply recommended normalization parameters (from `suggest_normalization_param`).  
  - Example: scaling intensity to `[0, 17]`.  
  - (optionally) downsample to match cytoplasm resolution -- required for aligning shapes of different channels  
- Apply Gaussian smoothing (e.g. σ = 1).  

4.2 **Segmentation**
- Use **AllenCell wrappers** (e.g. `dot_2d_slice_by_slice_wrapper` for spotty structures), based on the **lookup tables** found here:
  - General pipelines: https://www.allencell.org/segmenter.html#lookup-table 
  - Examples of notebooks with specific functions: https://github.com/AllenCell/aics-segmentation/tree/main/lookup_table_demo
  - Description of all modules available: https://allencell.github.io/aics-segmentation/aicssegmentation.core.html#   
- Returns binary masks (optionally cleaned by removing very small objects).  
- Save `.tif` masks and `.png` overlays for QC.  

4.3 **Quantification**
- Map segmented intracellular structures to corresponding cytoplasm masks.  
- Compute per-object and/or per-cell features:  
  - **count** (number of spots per cell)  
  - **volume** (µm³)  
  - **intensity** (mean intensity per object/cell)  
- Save results into CSV files for downstream analysis.

In [ ]:
# 4.0 obtain the suggested parameters for the preprocessing

from aicssegmentation.core.pre_processing_utils import suggest_normalization_param

### functions from aicssegmentation 
structure_img = all_volumes[0]["channels"]["intracellular"]
suggest_normalization_param(structure_img)


#### Example below: segmentation workflow for **spotty** structures 
[based on https://github.com/AllenCell/aics-segmentation/blob/main/lookup_table_demo/playground_spotty.ipynb ]:


About selected algorithms and tuned parameters

- **Intensity normalization**: Parameter intensity_scaling_param has two options: two values, say [A, B], or single value, say [K]. For the first case, A and B are non-negative values indicating that the full intensity range of the stack will first be cut-off into **[mean - A * std, mean + B * std]** and then rescaled to **[0, 1]**. The smaller the values of A and B are, the higher the contrast will be. For the second case, K>0 indicates min-max Normalization with an absolute intensity upper bound K (i.e., anything above K will be chopped off and reset as the minimum intensity of the stack) and K=0 means min-max Normalization without any intensity bound.

    - Parameter for fibrillarin: intensity_scaling_param = [0.5, 18]

    - Parameter for beta catenin: intensity_scaling_param = [4, 27]

- **Smoothing**

    3D gaussian smoothing with gaussian_smoothing_sigma = 1 (same for fibrillarin and beta catenin). The larger the value is, the more the image will be smoothed.


In [ ]:
# 4.1 do recommended pre-processing of the channel
##### params #####
gaussian_organelle = config["preprocessing"]["gaussian_organelle"]  # apply gaussian filter to nuclei channel
sigma_um_organelle = config["preprocessing"]["gaussian_sigma_organelle"]  # value of sigma for gaussian - could be adjusted depending on the data

### example: segmentation workflow for spotty structures [based on https://github.com/AllenCell/aics-segmentation/blob/main/lookup_table_demo/playground_spotty.ipynb]
# suggested parameter for normalization is [0.0, 17.0] (previous step)

from aicssegmentation.core.pre_processing_utils import intensity_normalization, image_smoothing_gaussian_3d

##### params #####
## note these refer to the aicssegmentation params, rather than our functions - bit confusing.
intensity_scaling_param = [0, 17]
gaussian_smoothing_sigma = 1

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name

    struct_img0 = v["channels"]["intracellular"]
    
    # --- preprocessing ---
    # intensity normalization
    struct_img_norm = intensity_normalization(struct_img0, scaling_param=intensity_scaling_param) # AllenCell function
    # making sure the image has the same shape as cytoplasm (from downsizing as the cytoplasm) -- TODO: needs fixing. remove exessive preprocessing, when no downsizing applied
    struct_img_norm = preprocess_3d_image(          
            struct_img_norm, 
            downsize_factor=downsize_factor,
            apply_gaussian_filter=gaussian_organelle,         
            voxel_size_um=voxel_size_um,        
            )    

    # smoothing with gaussian filter
    structure_img_smooth = image_smoothing_gaussian_3d(struct_img_norm, sigma=gaussian_smoothing_sigma) # # AllenCell function

    # saving results in a dictionary for future use
    if file_name not in results_dict:
        results_dict[file_name] = {}    # initialize dictionary for this file if it doesn't exist
    
    results_dict[file_name]["struct_img_norm"] = struct_img_norm
    results_dict[file_name]["structure_img_smooth"] = structure_img_smooth


In [ ]:
### View some masks - are they sensible?
## it's harder to see with subcellular objects - better to check the saved files.
# up to 5 pairs of (index, fname)
pairs = list(enumerate(results_dict.keys()))[:5]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 5))
if len(pairs) == 1:
    axes = [axes]

for ax, (idx, fname) in zip(axes, pairs):
    vol = all_volumes[idx]["channels"]
    mask = results_dict[fname]["struct_img_norm"]

    if len(vol.keys()) == 3:
        mid = len(vol["intracellular"]) // 2
        img = vol["intracellular"][mid]
        mask_slice = mask[mid]
    else:
        img = vol
        mask_slice = mask

    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(mask_slice == 0, mask_slice),
              cmap="autumn", alpha=0.5)
    ax.set_title(fname)
    ax.axis("off")

plt.tight_layout()
plt.show()

##### apply 2d spot filter
- Parameter syntax: [[scale_1, cutoff_1], [scale_2, cutoff_2], ....]

    - scale_x is set based on the estimated radius of your target spotty shape. For example, if visually the diameter of the spotty objects is usually 3~4 pixels, then you may want to set scale_x as 1 or something near 1 (like 1.25). Multiple scales can be used, if you have objects of very different sizes.
    - cutoff_x is a threshold applied on the actual filter reponse to get the binary result. Smaller cutoff_x may yielf fatter segmentation, while larger cutoff_x could be less permisive and yield less objects and slimmer segmentation.

Examples:
- Parameter for fibrillarin: s2_param = [[1, 0.01]]

- Parameter for beta catenin: s2_param = [[1.5, 0.01]]

In [ ]:
# 4.2 apply recommended algorithms to the channel (e.g. dot_2d_slice_by_slice_wrapper or filament_2d_wrapper for spotty structures) - AllenCell based
from aicssegmentation.core.seg_dot import dot_2d_slice_by_slice_wrapper
from skimage.morphology import remove_small_objects, binary_closing, ball , dilation

##### params #####
s2_param = [[0.75, 0.01]]
minArea = 5 # example of the smallest object size to detect

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name

    # retrieve the channel
    structure_img_smooth = results_dict[file_name]["structure_img_smooth"]
    
    # apply 2d spot filter (returns binary but not labelled mask)
    structure_mask = dot_2d_slice_by_slice_wrapper(structure_img_smooth, s2_param) # AllenCell function: could be substituted with other elements depending onthe data being processed (refer to lookup table)

    # post-process the image: remove small objects if needed
    #structure_mask = remove_small_objects(structure_mask>0, min_size=minArea, connectivity=1, in_place=False)


    #append and save results
    results_dict[file_name]["structure_mask"] = structure_mask

    # saving results into a mask (.tiff) as well as PNG overlay image
    save_segmentation_results(
        structure_img_smooth, 
        structure_mask, 
        output_root=output_folder, 
        experiment_label=f"{file_name}_structure",
        save_overlay=True
    )



In [ ]:
### View the same masks after filtering the small objects - are they sensible?
# up to 5 pairs of (index, fname)
pairs = list(enumerate(results_dict.keys()))[:5]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 5))
if len(pairs) == 1:
    axes = [axes]

for ax, (idx, fname) in zip(axes, pairs):
    vol = all_volumes[idx]["channels"]
    mask = results_dict[fname]["structure_mask"]

    if len(vol.keys()) == 3:
        mid = len(vol["intracellular"]) // 2
        img = vol["intracellular"][mid]
        mask_slice = mask[mid]
    else:
        img = vol
        mask_slice = mask

    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(mask_slice == 0, mask_slice),
              cmap="autumn", alpha=0.5)
    ax.set_title(fname)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 4.3 quantify the intracellular channel in respect to the cellular masks 
# TODO: add alternatives of quantifying per cell overall, nuclei only or cytoplasm only

import pandas as pd

##### params #####
return_level = 'both'   # whether to return results at the "object" level, "cell" level, or "both"


# --- analyse all images ---

all_results_cell = []  # store all per-volume results
all_results_obj = []

# loop over loaded images for segmentation
for v in all_volumes[:num_test_volumes]:
    file_name = v['filename']  # preserve original file name
    
    # --- get the corresponding masks and channels ---
    if file_name not in results_dict:
        raise KeyError(f"Masks for {file_name} not found in previous results.")
    
    #nuclei_mask = results_dict[file_name]["nuclei_mask"]       ### could be used in case calculation needs to be done per area of nucleus
    cytoplasm_mask = results_dict[file_name]["cytoplasm_mask"]
    structure_mask = results_dict[file_name]["structure_mask"]
    structure_img_smooth = results_dict[file_name]["structure_img_smooth"]


    # --- mapping intracellular structure channel to the previously obtained masks and quantifying ---
    df_obj, df_cells = quantify_structures_per_cell(
        cytoplasm_mask=cytoplasm_mask,      # cytoplasm_mask: labeled cells           TODO: add alternatives 
        object_mask=structure_mask,       # struct_mask: binary mask, e.g. from dot_2d_slice_by_slice_wrapper
        intensity_img=structure_img_smooth,     # struct_intensity: original preprocessed image used to detect spots
        voxel_size=voxel_size_um,
        features=("count","volume","intensity"), # which features to calculate
        return_level = return_level              # whether to return results at the "object" level, "cell" level, or "both" (in case of the latter: returns df_obj, df_cells)
    )

    # --- add metadata ---
     # add metadata: filename + space for experimental info if any
    df_cells["filename"] = file_name
    #df_obj["filename"] = file_name

    # --- store together results for all images ---
    all_results_cell.append(df_cells)
    #all_results_obj.append(df_obj)


# concatenate into one dataframe
df_all_cell = pd.concat(all_results_cell, ignore_index=True)
#df_all_obj = pd.concat(all_results_obj, ignore_index=True)

# TODO: choose which files to save dynamically (and update the path specification for each)

# save to csv
output_path_cell = os.path.join(output_folder, "cell_quantification_results.csv")
df_all_cell.to_csv(output_path_cell, index=False)

print(f"Results exported to {output_path_cell}")
print(df_all_cell.head())

#print(df_all_obj.head())

**Quality Control for Mapping**
- Ensure structure masks are **labeled** (not just binary).  
- Generate overlays mapping intracellular objects to their parent cell for manual validation  
- Export maximum-intensity projection overlays (`*_struct_to_cell_MIP.png`).

In [ ]:
# example quality check after you created df_obj (with 'label' and 'cell_id')
import numpy as np
from skimage.measure import label
from analysis_3d.scripts.quantification import save_struct_mip_overlay_by_cell

# make sure struct_mask is labeled
if np.array_equal(np.unique(structure_mask), [0,1]) or structure_mask.dtype == bool:
    struct_labeled = label(structure_mask)
else:
    struct_labeled = structure_mask.copy()

save_struct_mip_overlay_by_cell(
    struct_labeled=struct_labeled,      # NOT the cytoplasm mask
    df_obj=df_obj,
    raw_volume=structure_img_smooth,       # or original structure channel
    out_png=os.path.join(output_folder, f"{file_name}_struct_to_cell_MIP.png")
)
